# 훈련

In [40]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)
import xgboost as xgb
import lightgbm as lgb
import joblib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path

warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [41]:
def create_output_directory(dir_name='model'):
    """그래프 저장을 위한 디렉토리 생성"""
    output_dir = Path(dir_name)
    output_dir.mkdir(exist_ok=True)
    return output_dir

In [42]:
def load_and_preprocess_data(file_path):
    """데이터 로드 및 전처리"""
    print("=" * 50)
    print("데이터 로딩 중...")
    print("=" * 50)
    
    # CSV 파일 로드
    df = pd.read_csv(file_path, encoding='cp949')
    
    print(f"\n데이터셋 크기: {df.shape}")
    print(f"컬럼 목록: {df.columns.tolist()}")
    
    # date 컬럼 제거 
    if 'date' in df.columns:
        df = df.drop('date', axis=1)

    # MD_TQ 칼럼 제거
    if 'EX1.MD_TQ' in df.columns:
        df = df.drop('EX1.MD_TQ', axis=1)
    
    # 결측치 확인
    print(f"\n결측치 개수:\n{df.isnull().sum()}")
    
    # 결측치가 있는 행 제거
    df = df.dropna()
    
    # passorfail 컬럼의 분포 확인
    print("\n불량 분포:")
    print(df['passorfail'].value_counts())
    print(f"불량률: {df['passorfail'].sum() / len(df) * 100:.2f}%")
    
    return df

In [43]:
def prepare_train_test_data(df, test_size=0.3, random_state=42):
    """학습/테스트 데이터 분할"""
    print("\n" + "=" * 50)
    print("데이터 분할 중...")
    print("=" * 50)
    
    # Feature와 Target 분리
    X = df.drop('passorfail', axis=1)
    y = df['passorfail']
    
    # 학습/테스트 데이터 분할
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    print(f"\n학습 데이터 크기: {X_train.shape}")
    print(f"테스트 데이터 크기: {X_test.shape}")
    print(f"\n학습 데이터 불량 분포:\n{y_train.value_counts()}")
    print(f"학습 데이터 불량률: {y_train.sum() / len(y_train) * 100:.2f}%")
    print(f"\n테스트 데이터 불량 분포:\n{y_test.value_counts()}")
    
    
    return X_train, X_test, y_train, y_test


In [44]:
def train_decision_tree(X_train, y_train, max_depth=10, random_state=42):
    """Decision Tree 모델 학습"""
    print("\n" + "=" * 50)
    print("Decision Tree 모델 학습 중...")
    print("=" * 50)
    
    dt = DecisionTreeClassifier(max_depth=max_depth, random_state=random_state)
    dt.fit(X_train, y_train)
    
    print("Decision Tree 학습 완료!")
    return dt

In [45]:
def train_random_forest(X_train, y_train, n_estimators=100, max_depth=10, random_state=42):
    """Random Forest 모델 학습"""
    print("\n" + "=" * 50)
    print("Random Forest 모델 학습 중...")
    print("=" * 50)
    
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    
    print("Random Forest 학습 완료!")
    return rf

In [46]:
def train_adaboost(X_train, y_train, n_estimators=50, random_state=42):
    """AdaBoost 모델 학습"""
    print("\n" + "=" * 50)
    print("AdaBoost 모델 학습 중...")
    print("=" * 50)
    
    ada = AdaBoostClassifier(
        n_estimators=n_estimators,
        random_state=random_state,
        algorithm='SAMME'
    )
    ada.fit(X_train, y_train)
    
    print("AdaBoost 학습 완료!")
    return ada

In [47]:
def train_xgboost(X_train, y_train, n_estimators=100, max_depth=6, random_state=42):
    """XGBoost 모델 학습"""
    print("\n" + "=" * 50)
    print("XGBoost 모델 학습 중...")
    print("=" * 50)
    
    # 클래스 불균형 처리를 위한 scale_pos_weight 계산
    neg_count = (y_train == 0).sum()
    pos_count = (y_train == 1).sum()
    scale_pos_weight = neg_count / pos_count if pos_count > 0 else 1
    
    xgb_model = xgb.XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        random_state=random_state,
        eval_metric='logloss',
        use_label_encoder=False
    )
    xgb_model.fit(X_train, y_train)
    
    print("XGBoost 학습 완료!")
    return xgb_model

In [48]:
def train_lightgbm(X_train, y_train, n_estimators=100, max_depth=6, random_state=42):
    """LightGBM 모델 학습"""
    print("\n" + "=" * 50)
    print("LightGBM 모델 학습 중...")
    print("=" * 50)
    
    # 클래스 불균형 처리를 위한 scale_pos_weight 계산
    neg_count = (y_train == 0).sum()
    pos_count = (y_train == 1).sum()
    scale_pos_weight = neg_count / pos_count if pos_count > 0 else 1
    
    lgb_model = lgb.LGBMClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        random_state=random_state,
        verbose=-1
    )
    lgb_model.fit(X_train, y_train)
    
    print("LightGBM 학습 완료!")
    return lgb_model

In [49]:
def train_gradient_boosting(X_train, y_train, n_estimators=100, max_depth=5, random_state=42):
    """Gradient Boosting 모델 학습"""
    print("\n" + "=" * 50)
    print("Gradient Boosting 모델 학습 중...")
    print("=" * 50)
    
    gb = GradientBoostingClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=0.1,
        random_state=random_state
    )
    gb.fit(X_train, y_train)
    
    print("Gradient Boosting 학습 완료!")
    return gb

In [50]:
def train_logistic_regression(X_train, y_train, random_state=42):
    """Logistic Regression 모델 학습"""
    print("\n" + "=" * 50)
    print("Logistic Regression 모델 학습 중...")
    print("=" * 50)
    
    lr = LogisticRegression(
        max_iter=1000,
        random_state=random_state,
        class_weight='balanced'
    )
    lr.fit(X_train, y_train)
    
    print("Logistic Regression 학습 완료!")
    return lr

In [51]:
def train_dnn_model(X_train, y_train, hidden=256, dropout=0.2, learning_rate=0.001, epochs=200, batch_size=256):
    """DNN 모델 학습"""
    print("\n" + "=" * 50)
    print("DNN 모델 학습 중...")
    print("=" * 50)

    input_dim = X_train.shape[1]
    print(f"입력 변수 개수 (input_dim): {input_dim}")

    # 모델 정의
    keras.backend.clear_session()
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(hidden, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(2, activation='softmax')  # 0/1 클래스 분류용
    ])

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )

    # 클래스 불균형 자동 보정
    neg, pos = np.bincount(y_train)
    total = neg + pos
    class_weight = {0: total / (2.0 * neg), 1: total / (2.0 * pos)}
    print(f"클래스 가중치: {class_weight}")

    # 조기 종료 콜백
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=0
    )

    # 학습
    history = model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=epochs,
        batch_size=batch_size,
        class_weight=class_weight,
        callbacks=[early_stop],
        verbose=1
    )

    print("DNN 모델 학습 완료!")
    return model

In [52]:
def evaluate_model(model, X_test, y_test, model_name):
    """모델 평가 (DNN / ML 모델 통합 지원)"""
    print("\n" + "=" * 50)
    print(f"{model_name} 모델 평가")
    print("=" * 50)

    # 예측
    y_pred_raw = model.predict(X_test)

    # ① DNN 모델 (softmax → argmax 변환)
    if isinstance(y_pred_raw, np.ndarray) and y_pred_raw.ndim == 2 and y_pred_raw.shape[1] > 1:
        y_pred = np.argmax(y_pred_raw, axis=1)
        y_pred_proba = y_pred_raw[:, 1]

    # ② sklearn 모델 (예측이 1D, 예: [0, 1, 0, 1])
    elif y_pred_raw.ndim == 1:
        y_pred = y_pred_raw
        if hasattr(model, "predict_proba"):
            y_pred_proba = model.predict_proba(X_test)[:, 1]
        else:
            y_pred_proba = None

    # ③ 예외 처리 (혹시 모를 edge case)
    else:
        y_pred = np.round(y_pred_raw).astype(int)
        y_pred_proba = None

    # ROC-AUC 계산
    roc_auc = roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else 0

    # 평가 지표 계산
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    print(f"\nAccuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")

    print("\n분류 리포트:")
    print(classification_report(y_test, y_pred, target_names=['정상', '불량']))

    cm = confusion_matrix(y_test, y_pred)
    print("\nConfusion Matrix:")
    print(cm)

    return {
        "model_name": model_name,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "roc_auc": roc_auc,
        "confusion_matrix": cm,
        "y_pred_proba": y_pred_proba,
    }

In [53]:
def plot_feature_importance(model, feature_names, model_name, top_n=15, output_dir='model'):
    """Feature Importance 시각화"""
    if hasattr(model, 'feature_importances_'):
        importances = pd.Series(model.feature_importances_, index=feature_names)
        importances = importances.sort_values(ascending=False)
        
        plt.figure(figsize=(12, 8))
        importances.head(top_n).plot(kind='barh')
        plt.title(f'{model_name} - Top {top_n} Feature Importances')
        plt.xlabel('Importance')
        plt.ylabel('Features')
        plt.tight_layout()
        plt.savefig(f'{output_dir}/{model_name}_feature_importance.png', dpi=300, bbox_inches='tight')
        print(f"\n{output_dir}/{model_name} Feature Importance 저장 완료: {model_name}_feature_importance.png")
        plt.close()
        
        return importances

In [54]:
def plot_confusion_matrices(results, output_dir='model'):
    """모든 모델의 Confusion Matrix 시각화"""
    n_models = len(results)
    n_cols = 3
    n_rows = (n_models + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5*n_rows))
    axes = axes.flatten() if n_models > 1 else [axes]
    
    for idx, result in enumerate(results):
        sns.heatmap(
            result['confusion_matrix'],
            annot=True,
            fmt='d',
            cmap='Blues',
            ax=axes[idx],
            xticklabels=['정상', '불량'],
            yticklabels=['정상', '불량']
        )
        axes[idx].set_title(f"{result['model_name']}\nF1: {result['f1_score']:.4f} | ROC-AUC: {result['roc_auc']:.4f}")
        axes[idx].set_ylabel('실제')
        axes[idx].set_xlabel('예측')
    
    # 빈 서브플롯 제거
    for idx in range(n_models, len(axes)):
        fig.delaxes(axes[idx])
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/confusion_matrices.png', dpi=300, bbox_inches='tight')
    print(f"\n{output_dir}/confusion_matrices.png 저장 완료")
    plt.close()

In [55]:
def compare_models(results, output_dir='model'):
    """모델 성능 비교"""
    print("\n" + "=" * 50)
    print("모델 성능 비교")
    print("=" * 50)
    
    comparison_df = pd.DataFrame([
        {
            'Model': r['model_name'],
            'Accuracy': r['accuracy'],
            'Precision': r['precision'],
            'Recall': r['recall'],
            'F1-Score': r['f1_score'],
            'ROC-AUC': r['roc_auc']
        }
        for r in results
    ])
    
    # F1-Score로 정렬
    comparison_df = comparison_df.sort_values('F1-Score', ascending=False)
    
    print("\n", comparison_df.to_string(index=False))
    
    # 성능 비교 그래프
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
    cmap = plt.cm.get_cmap('tab10')
    colors = [cmap(i) for i in range(len(comparison_df))]
    
    for idx, metric in enumerate(metrics):
        ax = axes[idx // 3, idx % 3]
        bars = ax.bar(range(len(comparison_df)), comparison_df[metric], color=colors)
        ax.set_title(f'{metric} 비교', fontsize=14, fontweight='bold')
        ax.set_ylabel(metric, fontsize=12)
        ax.set_xlabel('Model', fontsize=12)
        ax.set_xticks(range(len(comparison_df)))
        ax.set_xticklabels(comparison_df['Model'], rotation=45, ha='right')
        ax.set_ylim([0, 1.05])
        ax.grid(axis='y', alpha=0.3)
        
        # 값 표시
        for i, bar in enumerate(bars):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.4f}',
                   ha='center', va='bottom', fontsize=9)
    
    # 종합 랭킹 표시
    ax = axes[1, 2]
    ranking_data = comparison_df[['Model', 'F1-Score', 'ROC-AUC']].head(5)
    ax.axis('tight')
    ax.axis('off')
    table = ax.table(cellText=ranking_data.values,
                     colLabels=ranking_data.columns,
                     cellLoc='center',
                     loc='center',
                     colWidths=[0.4, 0.3, 0.3])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    ax.set_title('Top 5 Models (F1-Score 기준)', fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/model_comparison.png', dpi=300, bbox_inches='tight')
    print(f"\n{output_dir}/model_comparison.png 저장 완료")
    plt.close()
    
    return comparison_df

In [56]:
def plot_roc_curves(results, y_test, output_dir='model'):
    """ROC Curve 시각화"""
    print("\nROC Curve 생성 중...")
    
    plt.figure(figsize=(12, 8))
    
    for result in results:
        if result['y_pred_proba'] is not None:
            fpr, tpr, _ = roc_curve(y_test, result['y_pred_proba'])
            plt.plot(fpr, tpr, label=f"{result['model_name']} (AUC = {result['roc_auc']:.4f})", linewidth=2)
    
    plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC Curves - 모델 비교', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right", fontsize=10)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/roc_curves.png', dpi=300, bbox_inches='tight')
    print(f"ROC Curves 저장 완료: {output_dir}/roc_curves.png")
    plt.close()

In [57]:
def save_models(models_dict, output_dir='model'):
    """학습된 모델 저장"""
    print("\n" + "=" * 50)
    print("모델 저장 중...")
    print("=" * 50)
    
    saved_files = []
    for name, model in models_dict.items():
        filename = f"{output_dir}/{name.lower().replace(' ', '_')}_model.pkl"
        joblib.dump(model, filename)
        saved_files.append(filename)
        print(f"  - {filename}")
    
    print("\n모델 저장 완료!")
    return saved_files

In [58]:
print("\n" + "=" * 50)
print("소성가공 압출공정 불량 탐지 모델 학습 시작")
print("=" * 50)

# 출력 디렉토리 생성
output_dir_path = create_output_directory('model')
output_dir = output_dir_path.absolute().as_posix()

# 1. 데이터 로드 및 전처리
df = load_and_preprocess_data('소성가공 압출공정 데이터셋.csv')

# 2. 학습/테스트 데이터 분할
X_train, X_test, y_train, y_test = prepare_train_test_data(df)

# 3. 여러 모델 학습
print("\n" + "=" * 50)
print("다양한 알고리즘으로 모델 학습 중...")
print("=" * 50)

models = {}
models['Decision Tree'] = train_decision_tree(X_train, y_train)
models['Random Forest'] = train_random_forest(X_train, y_train)
models['AdaBoost'] = train_adaboost(X_train, y_train)
models['XGBoost'] = train_xgboost(X_train, y_train)
models['LightGBM'] = train_lightgbm(X_train, y_train)
models['Gradient Boosting'] = train_gradient_boosting(X_train, y_train)
models['Logistic Regression'] = train_logistic_regression(X_train, y_train)
models['DNN'] = train_dnn_model(X_train, y_train)

# 4. 모델 평가
results = []
for name, model in models.items():
    results.append(evaluate_model(model, X_test, y_test, name))

# 5. Feature Importance 분석 (트리 기반 모델만)
feature_names = X_train.columns
for name, model in models.items():
    if name in ['Decision Tree', 'Random Forest', 'AdaBoost', 'XGBoost', 'LightGBM', 'Gradient Boosting']:
        plot_feature_importance(model, feature_names, name, output_dir=output_dir)

# 6. Confusion Matrix 시각화
plot_confusion_matrices(results, output_dir)

# 7. ROC Curve 시각화
plot_roc_curves(results, y_test, output_dir)

# 8. 모델 성능 비교
comparison_df = compare_models(results, output_dir)

# 9. 모델 저장
saved_files = save_models(models, output_dir)

# 10. 최종 결과 출력
print("\n" + "=" * 50)
print("학습 완료!")
print("=" * 50)
print("\n생성된 모델 파일:")
for file in saved_files:
    print(f"  - {file}")

print("\n생성된 시각화 파일:")
print("  - confusion_matrices.png")
print("  - model_comparison.png")
print("  - roc_curves.png")
for name in ['Decision Tree', 'Random Forest', 'AdaBoost', 'XGBoost', 'LightGBM', 'Gradient Boosting']:
    print(f"  - {name}_feature_importance.png")

# 최고 성능 모델 출력 (F1-Score 기준)
best_idx = comparison_df['F1-Score'].idxmax()
best_model = comparison_df.loc[best_idx]
print(f"\n{'='*50}")
print("🏆 최고 성능 모델 (F1-Score 기준)")
print(f"{'='*50}")
print(f"모델: {best_model['Model']}")
print(f"  - Accuracy:  {best_model['Accuracy']:.4f}")
print(f"  - Precision: {best_model['Precision']:.4f}")
print(f"  - Recall:    {best_model['Recall']:.4f}")
print(f"  - F1-Score:  {best_model['F1-Score']:.4f}")
print(f"  - ROC-AUC:   {best_model['ROC-AUC']:.4f}")

# Top 3 모델 출력
print(f"\n{'='*50}")
print("📊 Top 3 모델")
print(f"{'='*50}")
for i, (idx, row) in enumerate(comparison_df.head(3).iterrows(), 1):
    print(f"\n{i}. {row['Model']}")
    print(f"   F1-Score: {row['F1-Score']:.4f} | ROC-AUC: {row['ROC-AUC']:.4f}")



소성가공 압출공정 불량 탐지 모델 학습 시작
데이터 로딩 중...

데이터셋 크기: (17280, 20)
컬럼 목록: ['date', 'EX5.MELT_TEMP', 'EX4.MELT_TEMP', 'EX3.MELT_TEMP', 'EX2.MELT_TEMP', 'EX1.Z1_PV', 'EX1.Z2_PV', 'EX1.Z3_PV', 'EX1.Z4_PV', 'EX1.A1_PV', 'EX1.A2_PV', 'EX1.H1_PV', 'EX1.H2_PV', 'EX1.H3_PV', 'EX1.H4_PV', 'EX1.H2O_PV', 'EX1.MELT_P_PV', 'EX1.MD_PV', 'EX1.MD_TQ', 'passorfail']

결측치 개수:
EX5.MELT_TEMP     0
EX4.MELT_TEMP     0
EX3.MELT_TEMP     0
EX2.MELT_TEMP     0
EX1.Z1_PV         0
EX1.Z2_PV         0
EX1.Z3_PV         0
EX1.Z4_PV         0
EX1.A1_PV         0
EX1.A2_PV         0
EX1.H1_PV         0
EX1.H2_PV         0
EX1.H3_PV         0
EX1.H4_PV         0
EX1.H2O_PV        0
EX1.MELT_P_PV     0
EX1.MD_PV         0
passorfail       16
dtype: int64

불량 분포:
passorfail
0.0    17154
1.0      110
Name: count, dtype: int64
불량률: 0.64%

데이터 분할 중...

학습 데이터 크기: (12084, 17)
테스트 데이터 크기: (5180, 17)

학습 데이터 불량 분포:
passorfail
0.0    12007
1.0       77
Name: count, dtype: int64
학습 데이터 불량률: 0.64%

테스트 데이터 불량 분포:
passorfail
0.0    5

# 예측

In [59]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

In [60]:
def create_output_directory(dir_name='output'):
    """저장을 위한 디렉토리 생성"""
    output_dir = Path(dir_name)
    output_dir.mkdir(exist_ok=True)
    return output_dir

In [61]:
def load_model(model_path):
    """저장된 모델 로드"""
    if not Path(model_path).exists():
        raise FileNotFoundError(f"모델 파일을 찾을 수 없습니다: {model_path}")
    
    print(f"모델 로딩 중: {model_path}")
    model = joblib.load(model_path)
    print("모델 로딩 완료!")
    return model

In [62]:
def load_data(file_path):
    """예측할 데이터 로드"""
    print(f"\n데이터 로딩 중: {file_path}")
    df = pd.read_csv(file_path, encoding='cp949')
    
    # date 컬럼이 있으면 제거
    if 'date' in df.columns:
        dates = df['date'].copy()
        df = df.drop('date', axis=1)
    else:
        dates = None
    
        # MD_TQ 칼럼 제거
    if 'EX1.MD_TQ' in df.columns:
        df = df.drop('EX1.MD_TQ', axis=1)
    
    # passorfail 컬럼이 있으면 제거 (예측 데이터이므로)
    has_label = 'passorfail' in df.columns
    if has_label:
        true_labels = df['passorfail'].copy()
        df = df.drop('passorfail', axis=1)
    else:
        true_labels = None
    
    # 결측치 확인
    null_count = df.isnull().sum().sum()
    if null_count > 0:
        print(f"결측치 발견: {null_count}개")
        print("결측치가 있는 행 제거 중...")
        # 결측치가 있는 행의 인덱스 저장
        valid_idx = df.dropna().index
        df = df.loc[valid_idx]
        if dates is not None:
            dates = dates.loc[valid_idx]
        if true_labels is not None:
            true_labels = true_labels.loc[valid_idx]
        print(f"결측치 제거 후 데이터 크기: {df.shape}")
    
    print(f"데이터 크기: {df.shape}")
    return df, dates, true_labels, has_label

In [63]:
def predict(model, X, threshold=0.5):
    """불량 예측"""
    print("\n예측 수행 중...")
    
    # 확률 예측 (가능한 경우)
    if hasattr(model, 'predict_proba'):
        probabilities = model.predict_proba(X)
        defect_probs = probabilities[:, 1]  # 불량일 확률
        
        # 사용자 지정 임계값으로 예측
        if threshold != 0.5:
            print(f"사용자 지정 임계값 적용: {threshold}")
            predictions = (defect_probs >= threshold).astype(int)
        else:
            predictions = model.predict(X)
    else:
        predictions = model.predict(X)
        defect_probs = None
    
    print("예측 완료!")
    return predictions, defect_probs

In [64]:
def save_predictions(predictions, probs, dates, true_labels, output_path):
    """예측 결과 저장"""
    result_df = pd.DataFrame()
    
    if dates is not None:
        result_df['date'] = dates
    
    result_df['prediction'] = predictions
    result_df['prediction_label'] = result_df['prediction'].map({0: '정상', 1: '불량'})
    
    if probs is not None:
        result_df['defect_probability'] = probs
    
    if true_labels is not None:
        result_df['true_label'] = true_labels
        result_df['true_label_name'] = result_df['true_label'].map({0: '정상', 1: '불량'})
        result_df['correct'] = (result_df['prediction'] == result_df['true_label'])
    
    result_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n예측 결과 저장 완료: {output_path}")
    
    return result_df

In [65]:
def print_prediction_summary(predictions, true_labels=None, probs=None):
    """예측 결과 요약 출력"""
    print("\n" + "=" * 50)
    print("예측 결과 요약")
    print("=" * 50)
    
    total = len(predictions)
    defect_count = np.sum(predictions == 1)
    normal_count = np.sum(predictions == 0)
    
    print(f"\n총 예측 샘플 수: {total}")
    print(f"정상 예측: {normal_count} ({normal_count/total*100:.2f}%)")
    print(f"불량 예측: {defect_count} ({defect_count/total*100:.2f}%)")
    
    # 확률 분포 정보
    if probs is not None:
        print("\n불량 확률 통계:")
        print(f"  - 평균: {probs.mean():.4f}")
        print(f"  - 최대: {probs.max():.4f}")
        print(f"  - 최소: {probs.min():.4f}")
        
        # 위험도별 분류
        very_high_risk = np.sum(probs >= 0.9)
        high_risk = np.sum((probs >= 0.8) & (probs < 0.9))
        medium_risk = np.sum((probs >= 0.5) & (probs < 0.8))
        low_risk = np.sum(probs < 0.5)
        
        print("\n위험도별 분포:")
        print(f"  - 매우 높음 (≥90%): {very_high_risk}개 ({very_high_risk/total*100:.2f}%)")
        print(f"  - 높음 (80~90%):   {high_risk}개 ({high_risk/total*100:.2f}%)")
        print(f"  - 중간 (50~80%):   {medium_risk}개 ({medium_risk/total*100:.2f}%)")
        print(f"  - 낮음 (<50%):     {low_risk}개 ({low_risk/total*100:.2f}%)")
    
    if true_labels is not None:
        from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
        
        # NaN 체크 및 제거
        nan_count = pd.isna(true_labels).sum()
        if nan_count > 0:
            print(f"\n경고: true_labels에 {nan_count}개의 NaN 발견. 해당 행 제외하고 평가합니다.")
            valid_mask = ~pd.isna(true_labels)
            true_labels = true_labels[valid_mask]
            predictions = predictions[valid_mask]
            print(f"평가에 사용된 샘플 수: {len(predictions)}")
        
        accuracy = accuracy_score(true_labels, predictions)
        precision = precision_score(true_labels, predictions, zero_division=0)
        recall = recall_score(true_labels, predictions, zero_division=0)
        f1 = f1_score(true_labels, predictions, zero_division=0)
        cm = confusion_matrix(true_labels, predictions)
        
        print("\n" + "=" * 50)
        print("실제 레이블과 비교")
        print("=" * 50)
        print(f"\nAccuracy:  {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall:    {recall:.4f}")
        print(f"F1-Score:  {f1:.4f}")
        
        print(f"\nConfusion Matrix:")
        print(f"              예측_정상  예측_불량")
        print(f"실제_정상    {cm[0][0]:8d}   {cm[0][1]:8d}")
        print(f"실제_불량    {cm[1][0]:8d}   {cm[1][1]:8d}")

In [66]:
def print_high_risk_samples(result_df, top_n=10, threshold=0.8, output='high_risk_samples.csv'):
    """고위험 샘플 상세 정보 출력 및 CSV 저장"""
    if 'defect_probability' not in result_df.columns:
        return
    
    # 고위험 샘플 필터링
    high_risk = result_df[result_df['defect_probability'] >= threshold].copy()
    
    if len(high_risk) == 0:
        print(f"\n✅ 불량 확률 {threshold*100:.0f}% 이상인 고위험 샘플이 없습니다.")
        return
    
    # 확률 순으로 정렬
    high_risk = high_risk.sort_values('defect_probability', ascending=False)
    
    print("\n" + "=" * 70)
    print(f"⚠️  고위험 샘플 상세 정보 (불량 확률 ≥ {threshold*100:.0f}%)")
    print("=" * 70)
    
    print(f"\n총 고위험 샘플: {len(high_risk)}개")
    print(f"Top {min(top_n, len(high_risk))}개 샘플 상세:")
    print("-" * 70)
    
    # Top N 출력
    for i, (idx, row) in enumerate(high_risk.head(top_n).iterrows(), 1):
        print(f"\n[{i}] 샘플 정보:")
        if 'date' in row and pd.notna(row['date']):
            print(f"    날짜/시간: {row['date']}")
        print(f"    불량 확률: {row['defect_probability']:.2%} {'🔴' if row['defect_probability'] >= 0.95 else '🟠'}")
        print(f"    예측 결과: {row['prediction_label']}")
        if 'true_label_name' in row and pd.notna(row['true_label_name']):
            correct = "✅" if row.get('correct', False) else "❌"
            print(f"    실제 결과: {row['true_label_name']} {correct}")
    
    # 통계 요약
    print("\n" + "-" * 70)
    print("고위험 샘플 통계:")
    print(f"  - 평균 불량 확률: {high_risk['defect_probability'].mean():.2%}")
    print(f"  - 최대 불량 확률: {high_risk['defect_probability'].max():.2%}")
    print(f"  - 최소 불량 확률: {high_risk['defect_probability'].min():.2%}")
    
    # 정확도 (레이블이 있는 경우)
    if 'correct' in high_risk.columns:
        correct_count = high_risk['correct'].sum()
        total_count = len(high_risk)
        accuracy = correct_count / total_count
        print(f"  - 고위험 샘플 예측 정확도: {accuracy:.2%} ({correct_count}/{total_count})")
    
    # 저장된 파일 안내
    high_risk.to_csv(output, index=False, encoding='utf-8-sig')
    print(f"\n📄 고위험 샘플이 별도 파일로 저장되었습니다: {output}")
    
    # 콘솔에 전체 리스트 출력 (간단한 형식)
    print("\n" + "=" * 70)
    print("📋 고위험 샘플 전체 리스트 (콘솔 출력)")
    print("=" * 70)
    
    # 출력할 컬럼 선택 및 헤더 출력
    if 'date' in high_risk.columns:
        print(f"\n{'No.':<5} {'날짜/시간':<20} {'불량확률':<10} {'예측':<8} {'실제':<8} {'정확도':<6}")
    else:
        print(f"\n{'No.':<5} {'불량확률':<10} {'예측':<8} {'실제':<8} {'정확도':<6}")
    print("-" * 70)
    
    # 전체 고위험 샘플 출력
    for i, (idx, row) in enumerate(high_risk.iterrows(), 1):
        date_str = str(row['date'])[:20] if 'date' in row and pd.notna(row['date']) else ""
        prob_str = f"{row['defect_probability']:.2%}"
        pred_str = "불량" if row['prediction'] == 1 else "정상"
        
        if 'true_label_name' in row and pd.notna(row['true_label_name']):
            true_str = str(row['true_label_name'])
            correct_str = "✅" if row.get('correct', False) else "❌"
        else:
            true_str = "N/A"
            correct_str = ""
        
        if 'date' in high_risk.columns:
            print(f"{i:<5} {date_str:<20} {prob_str:<10} {pred_str:<8} {true_str:<8} {correct_str:<6}")
        else:
            print(f"{i:<5} {prob_str:<10} {pred_str:<8} {true_str:<8} {correct_str:<6}")
    
    print("=" * 70)

In [67]:
def list_available_models():
    """사용 가능한 모델 파일 목록 출력"""
    import glob
    model_files = glob.glob('*_model.pkl')
    if model_files:
        print("\n사용 가능한 모델:")
        for i, model_file in enumerate(model_files, 1):
            print(f"  {i}. {model_file}")
    return model_files

In [68]:
model_dir = "model"
model_path = f'{model_dir}/xgboost_model.pkl'
data_path = "소성가공 압출공정 데이터셋.csv"
threshold = 0.5
output_dir = "output"
output_path = f"{output_dir}/prediction_results.csv"
high_risk_output_path = f"{output_dir}/high_risk_result.csv"
top_n = 15
high_risk_threshold = 0.8

In [69]:
print("=" * 50)
print("소성가공 압출공정 불량 예측")
print("=" * 50)

# 출력 디렉토리 생성
create_output_directory(output_dir)

# 1. 모델 로드
model = load_model(model_path)

# 모델 정보 출력
model_name = model_path.replace('_model.pkl', '').replace('_', ' ').title()
print(f"\n사용 모델: {model_name}")
if threshold != 0.5:
    print(f"불량 판정 임계값: {threshold}")

# 2. 데이터 로드
X, dates, true_labels, has_label = load_data(data_path)

# 3. 예측 수행
predictions, probs = predict(model, X, threshold)

# 4. 결과 저장
result_df = save_predictions(predictions, probs, dates, true_labels, output_path)

# 5. 결과 요약 출력
print_prediction_summary(predictions, true_labels, probs)

# 6. 고위험 샘플 상세 정보 출력
print_high_risk_samples(result_df, top_n, high_risk_threshold, high_risk_output_path)

# 7. 간단 경고 메시지
if probs is not None:
    high_risk_count = np.sum(probs >= high_risk_threshold)
    if high_risk_count > 0:
        print(f"\n⚠️  총 {high_risk_count}개의 고위험 샘플이 발견되었습니다!")
        print(f"   위 상세 정보 및 '{high_risk_output_path}' 파일을 확인하세요.")

print("\n" + "=" * 50)
print("예측 완료!")
print("=" * 50)
print(f"\n📄 예측 결과 파일: {output_path}")

소성가공 압출공정 불량 예측
모델 로딩 중: model/xgboost_model.pkl
모델 로딩 완료!

사용 모델: Model/Xgboost

데이터 로딩 중: 소성가공 압출공정 데이터셋.csv
데이터 크기: (17280, 17)

예측 수행 중...
예측 완료!

예측 결과 저장 완료: output/prediction_results.csv

예측 결과 요약

총 예측 샘플 수: 17280
정상 예측: 17162 (99.32%)
불량 예측: 118 (0.68%)

불량 확률 통계:
  - 평균: 0.0069
  - 최대: 0.9999
  - 최소: 0.0000

위험도별 분포:
  - 매우 높음 (≥90%): 113개 (0.65%)
  - 높음 (80~90%):   1개 (0.01%)
  - 중간 (50~80%):   4개 (0.02%)
  - 낮음 (<50%):     17162개 (99.32%)

경고: true_labels에 16개의 NaN 발견. 해당 행 제외하고 평가합니다.
평가에 사용된 샘플 수: 17264

실제 레이블과 비교

Accuracy:  0.9994
Precision: 0.9237
Recall:    0.9909
F1-Score:  0.9561

Confusion Matrix:
              예측_정상  예측_불량
실제_정상       17145          9
실제_불량           1        109

⚠️  고위험 샘플 상세 정보 (불량 확률 ≥ 80%)

총 고위험 샘플: 114개
Top 15개 샘플 상세:
----------------------------------------------------------------------

[1] 샘플 정보:
    날짜/시간: 2020 10 30 13:28:24
    불량 확률: 99.99% 🔴
    예측 결과: 불량
    실제 결과: 불량 ✅

[2] 샘플 정보:
    날짜/시간: 2020 10 30 13:28:19
    불량 확률: 99.99% 

# 변수 특징

In [70]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [71]:
def create_output_directory(dir_name='trend_graphs'):
    """그래프 저장을 위한 디렉토리 생성"""
    output_dir = Path(dir_name)
    output_dir.mkdir(exist_ok=True)
    return output_dir

In [72]:
def load_data(file_path):
    """데이터 로드"""
    print(f"데이터 로딩 중: {file_path}")
    df = pd.read_csv(file_path)
    print(f"데이터 형태: {df.shape}")
    
    # 결측치 정보 출력
    missing_count = df.isnull().sum()
    if missing_count.sum() > 0:
        print(f"\n결측치 정보:")
        missing_cols = missing_count[missing_count > 0]
        for col, count in missing_cols.items():
            print(f"  - {col}: {count}개 ({count/len(df)*100:.2f}%)")
    else:
        print("\n결측치 없음")
    
    print(f"\nPassorFail 값 분포:\n{df['passorfail'].value_counts()}")
    return df

In [73]:
def plot_variable_trends_by_class(df, output_dir):
    """PassorFail 값에 따른 각 변수의 추이 시각화"""
    # PassorFail 값별로 데이터 분리 및 인덱스 리셋
    pass_data = df[df['passorfail'] == 0].reset_index(drop=True)
    fail_data = df[df['passorfail'] == 1].reset_index(drop=True)
    
    # 숫자형 컬럼만 선택 (date와 passorfail 제외)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols.remove('passorfail')
    
    print(f"\n시각화할 변수 개수: {len(numeric_cols)}")
    print(f"Pass 샘플 수: {len(pass_data)}, Fail 샘플 수: {len(fail_data)}")
    
    # 각 변수에 대한 시계열 플롯
    for col in numeric_cols:
        # 결측치 제거
        pass_data_clean = pass_data[col].dropna()
        fail_data_clean = fail_data[col].dropna()
        
        # 결측치 제거 후 데이터가 없으면 스킵
        if len(pass_data_clean) == 0 and len(fail_data_clean) == 0:
            print(f"⚠ {col} - 모든 데이터가 결측치입니다. 스킵합니다.")
            continue
        
        plt.figure(figsize=(15, 6))
        
        # Pass(1) 데이터 플롯 - 0부터 시작하는 새로운 인덱스 사용
        plt.subplot(1, 2, 1)
        if len(pass_data_clean) > 0:
            plt.plot(range(len(pass_data_clean)), pass_data_clean, 
                    alpha=0.7, linewidth=0.5, color='green')
            plt.title(f'{col} - Pass (정상) [N={len(pass_data_clean)}]', fontsize=12, fontweight='bold')
        else:
            plt.title(f'{col} - Pass (정상) [데이터 없음]', fontsize=12, fontweight='bold')
            plt.text(0.5, 0.5, '결측치로 인해\n데이터가 없습니다', 
                    ha='center', va='center', transform=plt.gca().transAxes, fontsize=14)
        plt.xlabel('샘플 인덱스 (0부터 시작)')
        plt.ylabel(col)
        plt.grid(True, alpha=0.3)
        
        # Fail(0) 데이터 플롯 - 0부터 시작하는 새로운 인덱스 사용
        plt.subplot(1, 2, 2)
        if len(fail_data_clean) > 0:
            plt.plot(range(len(fail_data_clean)), fail_data_clean, 
                    alpha=0.7, linewidth=0.5, color='red')
            plt.title(f'{col} - Fail (불량) [N={len(fail_data_clean)}]', fontsize=12, fontweight='bold')
        else:
            plt.title(f'{col} - Fail (불량) [데이터 없음]', fontsize=12, fontweight='bold')
            plt.text(0.5, 0.5, '결측치로 인해\n데이터가 없습니다', 
                    ha='center', va='center', transform=plt.gca().transAxes, fontsize=14)
        plt.xlabel('샘플 인덱스 (0부터 시작)')
        plt.ylabel(col)
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        # 파일명에서 특수문자 제거
        safe_col_name = col.replace('.', '_').replace('/', '_')
        plt.savefig(output_dir / f'trend_{safe_col_name}.png', dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"✓ {col} 그래프 저장 완료")

In [74]:
def plot_variable_distribution_comparison(df, output_dir):
    """PassorFail 값에 따른 변수 분포 비교"""
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols.remove('passorfail')
    
    print("\n분포 비교 그래프 생성 중...")
    
    for col in numeric_cols:
        # 결측치 제거
        pass_data_clean = df[df['passorfail'] == 0][col].dropna()
        fail_data_clean = df[df['passorfail'] == 1][col].dropna()
        
        # 결측치 제거 후 데이터가 없으면 스킵
        if len(pass_data_clean) == 0 and len(fail_data_clean) == 0:
            print(f"⚠ {col} - 모든 데이터가 결측치입니다. 스킵합니다.")
            continue
        
        plt.figure(figsize=(12, 5))
        
        # 히스토그램 비교
        plt.subplot(1, 2, 1)
        if len(pass_data_clean) > 0:
            plt.hist(pass_data_clean, bins=50, alpha=0.6, 
                    label='Pass (정상)', color='green', density=True)
        if len(fail_data_clean) > 0:
            plt.hist(fail_data_clean, bins=50, alpha=0.6, 
                    label='Fail (불량)', color='red', density=True)
        plt.xlabel(col)
        plt.ylabel('밀도')
        plt.title(f'{col} - 분포 비교', fontweight='bold')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # 박스플롯 비교
        plt.subplot(1, 2, 2)
        data_to_plot = []
        labels = []
        if len(pass_data_clean) > 0:
            data_to_plot.append(pass_data_clean)
            labels.append('Pass (정상)')
        if len(fail_data_clean) > 0:
            data_to_plot.append(fail_data_clean)
            labels.append('Fail (불량)')
        
        if len(data_to_plot) > 0:
            bp = plt.boxplot(data_to_plot, patch_artist=True)
            for i, box in enumerate(bp['boxes']):
                box.set_facecolor('green' if 'Pass' in labels[i] else 'red')
            plt.xticks(range(1, len(labels) + 1), labels)
        plt.ylabel(col)
        plt.title(f'{col} - 박스플롯 비교', fontweight='bold')
        plt.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        
        safe_col_name = col.replace('.', '_').replace('/', '_')
        plt.savefig(output_dir / f'distribution_{safe_col_name}.png', dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"✓ {col} 분포 그래프 저장 완료")

In [75]:
def plot_overall_comparison(df, output_dir):
    """전체 변수들의 평균값 비교"""
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols.remove('passorfail')
    
    # Pass와 Fail의 평균값 계산 (결측치 제거)
    pass_means = df[df['passorfail'] == 1][numeric_cols].mean(skipna=True)
    fail_means = df[df['passorfail'] == 0][numeric_cols].mean(skipna=True)
    
    # 차이가 큰 순서로 정렬
    mean_diff = abs(pass_means - fail_means)
    sorted_cols = mean_diff.sort_values(ascending=False).index
    
    # 상위 20개 변수만 표시
    top_n = min(20, len(sorted_cols))
    top_cols = sorted_cols[:top_n]
    
    plt.figure(figsize=(14, 8))
    x = np.arange(len(top_cols))
    width = 0.35
    
    plt.bar(x - width/2, pass_means[top_cols], width, label='Pass (정상)', 
           color='green', alpha=0.7)
    plt.bar(x + width/2, fail_means[top_cols], width, label='Fail (불량)', 
           color='red', alpha=0.7)
    
    plt.xlabel('변수')
    plt.ylabel('평균값')
    plt.title('PassorFail 값에 따른 변수별 평균값 비교 (차이가 큰 상위 20개)', 
             fontsize=14, fontweight='bold')
    plt.xticks(x, top_cols, rotation=45, ha='right')
    plt.legend()
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    
    plt.savefig(output_dir / 'overall_mean_comparison.png', dpi=200, bbox_inches='tight')
    plt.close()
    
    print("\n✓ 전체 평균값 비교 그래프 저장 완료")

In [76]:
def plot_correlation_heatmap(df, output_dir):
    """PassorFail별 상관관계 히트맵"""
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    # Pass 데이터 상관관계
    pass_corr = df[df['passorfail'] == 1][numeric_cols].corr()
    sns.heatmap(pass_corr, ax=axes[0], cmap='coolwarm', center=0, 
               square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
    axes[0].set_title('상관관계 히트맵 - Pass (정상)', fontsize=14, fontweight='bold')
    
    # Fail 데이터 상관관계
    fail_corr = df[df['passorfail'] == 0][numeric_cols].corr()
    sns.heatmap(fail_corr, ax=axes[1], cmap='coolwarm', center=0, 
               square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
    axes[1].set_title('상관관계 히트맵 - Fail (불량)', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(output_dir / 'correlation_heatmap.png', dpi=200, bbox_inches='tight')
    plt.close()
    
    print("✓ 상관관계 히트맵 저장 완료")

In [77]:
print("="*60)
print("PassorFail 값에 따른 변수 추이 시각화 시작")
print("="*60)

# 데이터 로드
df = load_data('소성가공 압출공정 데이터셋.csv')

# 출력 디렉토리 생성
output_dir = create_output_directory('trend_graphs')
print(f"\n그래프 저장 디렉토리: {output_dir.absolute()}")

# 1. 전체 평균값 비교
print("\n" + "="*60)
print("1. 전체 평균값 비교 그래프 생성")
print("="*60)
plot_overall_comparison(df, output_dir)

# 2. 상관관계 히트맵
print("\n" + "="*60)
print("2. 상관관계 히트맵 생성")
print("="*60)
plot_correlation_heatmap(df, output_dir)

# 3. 변수별 추이 그래프
print("\n" + "="*60)
print("3. 변수별 시계열 추이 그래프 생성")
print("="*60)
plot_variable_trends_by_class(df, output_dir)

# 4. 변수별 분포 비교 그래프
print("\n" + "="*60)
print("4. 변수별 분포 비교 그래프 생성")
print("="*60)
plot_variable_distribution_comparison(df, output_dir)

print("\n" + "="*60)
print("모든 그래프 생성 완료!")
print(f"저장 위치: {output_dir.absolute()}")
print("="*60)

PassorFail 값에 따른 변수 추이 시각화 시작
데이터 로딩 중: 소성가공 압출공정 데이터셋.csv
데이터 형태: (17280, 20)

결측치 정보:
  - passorfail: 16개 (0.09%)

PassorFail 값 분포:
passorfail
0.0    17154
1.0      110
Name: count, dtype: int64

그래프 저장 디렉토리: v:\dev\python\kamp\trend_graphs

1. 전체 평균값 비교 그래프 생성

✓ 전체 평균값 비교 그래프 저장 완료

2. 상관관계 히트맵 생성
✓ 상관관계 히트맵 저장 완료

3. 변수별 시계열 추이 그래프 생성

시각화할 변수 개수: 18
Pass 샘플 수: 17154, Fail 샘플 수: 110
✓ EX5.MELT_TEMP 그래프 저장 완료
✓ EX4.MELT_TEMP 그래프 저장 완료
✓ EX3.MELT_TEMP 그래프 저장 완료
✓ EX2.MELT_TEMP 그래프 저장 완료
✓ EX1.Z1_PV 그래프 저장 완료
✓ EX1.Z2_PV 그래프 저장 완료
✓ EX1.Z3_PV 그래프 저장 완료
✓ EX1.Z4_PV 그래프 저장 완료
✓ EX1.A1_PV 그래프 저장 완료
✓ EX1.A2_PV 그래프 저장 완료
✓ EX1.H1_PV 그래프 저장 완료
✓ EX1.H2_PV 그래프 저장 완료
✓ EX1.H3_PV 그래프 저장 완료
✓ EX1.H4_PV 그래프 저장 완료
✓ EX1.H2O_PV 그래프 저장 완료
✓ EX1.MELT_P_PV 그래프 저장 완료
✓ EX1.MD_PV 그래프 저장 완료
✓ EX1.MD_TQ 그래프 저장 완료

4. 변수별 분포 비교 그래프 생성

분포 비교 그래프 생성 중...
✓ EX5.MELT_TEMP 분포 그래프 저장 완료
✓ EX4.MELT_TEMP 분포 그래프 저장 완료
✓ EX3.MELT_TEMP 분포 그래프 저장 완료
✓ EX2.MELT_TEMP 분포 그래프 저장 완료
✓ EX1.Z1_PV 분포 그래프 저장 완료
✓ EX1.Z2_PV